# Bi-Mamba 55M — Chinese ↔ Vietnamese translation, end-to-end on Colab

Notebook chạy **toàn bộ pipeline** từ A đến Z: tải dữ liệu → train tokenizer → train mô hình Bi-Mamba 55M → đánh giá SacreBLEU → dịch demo.

**Yêu cầu:** Runtime → GPU (T4 / L4 / A100 đều OK). Bộ nhớ RAM ≥ 12 GB.

Toàn bộ tham số có thể chỉnh ở ô "Configuration". Mặc định subsample còn 200K cặp câu để 1 lần chạy đầy đủ trên T4 free trong khoảng 1.5–2 giờ; bỏ subsample nếu muốn train trên toàn bộ ~1M cặp.

## 1. Lấy mã nguồn

**Cách 1 (khuyên dùng):** clone từ GitHub.

```python
!git clone https://github.com/<your-username>/bi-mamba-zh-vi.git
%cd bi-mamba-zh-vi
```

**Cách 2:** upload thư mục lên Colab (Files → Upload folder) rồi `%cd bi-mamba-zh-vi`.

**Cách 3:** mount Google Drive nếu bạn đã đặt repo ở Drive.

In [ ]:
# Mount Google Drive (optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)

In [ ]:
# Clone repo if not already there. Replace URL with your own fork if you push it.
import os
REPO_URL = 'https://github.com/ChauDucToan/bi-mamba-zh-vi.git'  # <-- chỉnh nếu cần
REPO_DIR = '/content/bi-mamba-zh-vi'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR} || echo 'Clone failed - upload the project folder manually instead.'
if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
!pwd && ls

## 2. Cài đặt dependencies

Mamba dùng kernel CUDA tăng tốc rất nhiều. `mamba-ssm` và `causal-conv1d` cài qua pip wheel. Nếu cài lỗi, repo có **fallback PyTorch thuần** — vẫn chạy được nhưng chậm hơn 5–10×.

In [ ]:
!pip install -q -r requirements.txt
# Try to install the optional fast kernels. Skip silently if it fails.
!pip install -q causal-conv1d mamba-ssm 2>/dev/null || echo 'mamba-ssm not installed; using pure-PyTorch fallback (slower).'
import torch, sys
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
sys.path.insert(0, 'src')

## 3. Configuration

Sửa các giá trị bên dưới nếu muốn chạy nhanh / lâu hơn. Mặc định: subsample 200K cặp, max_steps 30K — phù hợp T4 trong 1.5–2 giờ.

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('configs/bi_mamba_55m.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# === Quick-tune knobs ===
cfg['data']['max_train_pairs'] = 200_000   # set to None to use full corpus
cfg['data']['max_valid_pairs'] = 2_000
cfg['data']['max_test_pairs']  = 2_000
cfg['train']['max_steps']      = 30_000
cfg['train']['batch_size']     = 32        # tăng lên 64+ nếu có A100
cfg['train']['amp_dtype']      = 'bf16'    # 'fp16' nếu T4 không hỗ trợ bf16
cfg['train']['eval_every']     = 2_000
cfg['train']['save_every']     = 2_000
cfg['train']['log_every']      = 50
cfg['train']['num_workers']    = 2

# Lưu config đã chỉnh
Path('configs/_colab.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print('Saved configs/_colab.yaml')

## 4. Tải + chuẩn bị dữ liệu (OPUS-100 zh-vi)

Sẽ tải `Helsinki-NLP/opus-100` config `vi-zh` (~1M cặp), lọc theo độ dài và lưu JSONL. Dùng cờ `--max-train-pairs` để subsample.

Bạn cũng có thể dùng dataset riêng: thêm `--custom-jsonl path/to/your.jsonl` trong đó mỗi dòng là `{"zh": "…", "vi": "…"}`.

In [ ]:
!python scripts/prepare_data.py --config configs/_colab.yaml

## 5. Train SentencePiece tokenizer (chia sẻ chung zh+vi)

In [ ]:
!python scripts/train_tokenizer.py --config configs/_colab.yaml

## 6. Khởi tạo model + kiểm tra số tham số

In [ ]:
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.utils import count_parameters, human_format

model = BiMambaTranslator(ModelConfig(**cfg['model']))
n = count_parameters(model)
print(f'Bi-Mamba parameters: {human_format(n)}  ({n:,})')

## 7. Train end-to-end

Checkpoint sẽ được ghi vào `runs/bi_mamba_55m/`. Có thể tắt và bật lại notebook với `--resume <ckpt>` để tiếp tục train.

In [ ]:
!python scripts/train.py --config configs/_colab.yaml

## 8. Đánh giá SacreBLEU + chrF cả hai chiều

In [ ]:
!python scripts/evaluate.py --config configs/_colab.yaml --num-samples 1000 --beam-size 4

## 9. Demo dịch

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(f"{cfg['train']['output_dir']}/latest.pt", map_location='cpu')
model = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
model.load_state_dict(ckpt['model']); model.eval()
tok = Tokenizer(f"{cfg['data']['tokenizer_dir']}/spm.model")

zh_samples = [
    '你好，世界。',
    '今天的天气真好。',
    '这本书我已经读完了。',
]
vi_samples = [
    'Xin chào thế giới.',
    'Hôm nay trời thật đẹp.',
    'Tôi đã đọc xong cuốn sách này.',
]
print('--- zh → vi ---')
for s, t in zip(zh_samples, translate_batch(model, tok, zh_samples, 'zh2vi', beam_size=4, device=device)):
    print(f'  {s!r} -> {t!r}')
print('--- vi → zh ---')
for s, t in zip(vi_samples, translate_batch(model, tok, vi_samples, 'vi2zh', beam_size=4, device=device)):
    print(f'  {s!r} -> {t!r}')

## 10. (Tuỳ chọn) Lưu checkpoint vào Drive

In [ ]:
import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/bi_mamba_55m'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for fname in ('latest.pt', 'final.pt', 'config.yaml'):
        src = f"{cfg['train']['output_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, DRIVE_DIR)
            print('Copied', src, '->', DRIVE_DIR)
    # Tokenizer
    tok_dst = f'{DRIVE_DIR}/tokenizer'
    os.makedirs(tok_dst, exist_ok=True)
    for fname in ('spm.model', 'spm.vocab'):
        src = f"{cfg['data']['tokenizer_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, tok_dst)
    print('Done. Files in', DRIVE_DIR)
else:
    print('Drive not mounted; skipping.')